# EconDash reader in a LlamaIndex vector index

One document per indicator and country. A licence that forbids redistribution
is skipped. The embedding below is a token counter, so the notebook runs on the
free trial without an embedding API. Swap in your own `embed_model` for production.


In [ ]:
import hashlib

from llama_index.core import VectorStoreIndex
from llama_index.core.base.embeddings.base import BaseEmbedding
from llama_index.tools.econdash import EconDashReader


class WordEmbedding(BaseEmbedding):
    def _vector(self, text: str) -> list[float]:
        vector = [0.0] * 64
        for token in text.lower().split():
            slot = int(hashlib.md5(token.encode()).hexdigest(), 16) % 64
            vector[slot] += 1.0
        return vector

    def _get_text_embedding(self, text: str) -> list[float]:
        return self._vector(text)

    def _get_query_embedding(self, query: str) -> list[float]:
        return self._vector(query)

    async def _aget_query_embedding(self, query: str) -> list[float]:
        return self._vector(query)


docs = EconDashReader().load_data(
    indicators=["gdp-nominal", "unemployment-rate-u3"],
    countries=["USA", "DEU"],
)
print(f"{len(docs)} documents")
for doc in docs:
    print(doc.metadata)

index = VectorStoreIndex.from_documents(docs, embed_model=WordEmbedding())
hits = index.as_retriever(similarity_top_k=1).retrieve("United States GDP nominal")
print(hits[0].metadata)
print(hits[0].text[:500])